# Silver - Limpieza y enriquecimiento

Parte 3 del proyecto: leer Bronze, tipar datos, eliminar duplicados y generar variables derivadas por ventanas temporales.

In [12]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("silver-payments-job")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.6.1"
        ])
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "admin123")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.silver")
print("Spark e Iceberg listos")

Spark e Iceberg listos


## 1) Lectura de Bronze y conversión de tipos

In [13]:
from pyspark.sql import functions as F

bronze_df = spark.table("lakehouse.bronze.payments_raw")

typed_df = (
    bronze_df
    .withColumn("event_ts", F.to_timestamp("event_time"))
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn("country", F.upper(F.col("country")))
    .withColumn("currency", F.upper(F.col("currency")))
    .withColumn("status", F.lower(F.col("status")))
)

typed_df.select("event_time", "event_ts", "amount", "country", "currency", "status").show(5, truncate=False)

+---------------------------+--------------------------+------+-------+--------+--------+
|event_time                 |event_ts                  |amount|country|currency|status  |
+---------------------------+--------------------------+------+-------+--------+--------+
|2026-03-24T16:05:14.040326Z|2026-03-24 16:05:14.040326|195.76|CI     |EUR     |approved|
|2026-03-24T16:05:49.350245Z|2026-03-24 16:05:49.350245|484.00|BN     |USD     |approved|
|2026-03-24T16:05:46.445727Z|2026-03-24 16:05:46.445727|450.84|TD     |USD     |approved|
|2026-03-24T16:05:20.774484Z|2026-03-24 16:05:20.774484|474.66|AE     |GBP     |declined|
|2026-03-24T16:05:25.589938Z|2026-03-24 16:05:25.589938|401.14|MG     |AUD     |declined|
+---------------------------+--------------------------+------+-------+--------+--------+
only showing top 5 rows



## 2) Eliminación de duplicados por payment_id (último evento)

In [14]:
from pyspark.sql.window import Window

w_dedup = Window.partitionBy("payment_id").orderBy(F.col("event_ts").desc_nulls_last())

dedup_df = (
    typed_df
    .withColumn("rn", F.row_number().over(w_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print(f"Registros bronze: {bronze_df.count()}")
print(f"Registros tras deduplicar: {dedup_df.count()}")

Registros bronze: 179
Registros tras deduplicar: 179


## 3) Enriquecimiento con ventanas temporales

In [15]:
base = dedup_df.withColumn("event_ts_seconds", F.col("event_ts").cast("long"))

w_card_5m = Window.partitionBy("card_id").orderBy("event_ts_seconds").rangeBetween(-300, 0)
w_card_10m = Window.partitionBy("card_id").orderBy("event_ts_seconds").rangeBetween(-600, 0)
w_card_1h = Window.partitionBy("card_id").orderBy("event_ts_seconds").rangeBetween(-3600, 0)
w_device_1h = Window.partitionBy("device_id").orderBy("event_ts_seconds").rangeBetween(-3600, 0)

silver_df = (
    base
    .withColumn("tx_count_card_5m", F.count("payment_id").over(w_card_5m))
    .withColumn("distinct_merchants_card_10m", F.approx_count_distinct("merchant_id").over(w_card_10m))
    .withColumn("distinct_countries_card_1h", F.approx_count_distinct("country").over(w_card_1h))
    .withColumn("distinct_cards_device_1h", F.approx_count_distinct("card_id").over(w_device_1h))
    .withColumn(
        "declined_ratio_card_1h",
        F.avg(F.when(F.col("status") == "declined", F.lit(1.0)).otherwise(F.lit(0.0))).over(w_card_1h)
    )
    .drop("event_ts_seconds")
)

silver_df.select(
    "payment_id",
    "card_id",
    "event_ts",
    "tx_count_card_5m",
    "distinct_merchants_card_10m",
    "distinct_countries_card_1h",
    "distinct_cards_device_1h",
    "declined_ratio_card_1h"
).show(10, truncate=False)

+--------------------------------------------+----------+--------------------------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|payment_id                                  |card_id   |event_ts                  |tx_count_card_5m|distinct_merchants_card_10m|distinct_countries_card_1h|distinct_cards_device_1h|declined_ratio_card_1h|
+--------------------------------------------+----------+--------------------------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|1af6c73f-cbc9-4e21-b5dd-823b3afc2e13        |CARD_00002|2026-03-24 16:04:54.681398|1               |1                          |1                         |1                       |0.0                   |
|ef128b22-3c83-4ab3-90be-4016f86b6640_retry_0|CARD_00002|2026-03-24 16:05:25.589938|2               |2                          |2                         |2                       

## 4) Escritura a Iceberg Silver

In [16]:
(
    silver_df.writeTo("lakehouse.silver.payments_enriched")
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .createOrReplace()
)

print("Tabla Silver actualizada: lakehouse.silver.payments_enriched")

Tabla Silver actualizada: lakehouse.silver.payments_enriched


## 5) Validación rápida de resultados

In [17]:
spark.sql("SELECT COUNT(*) AS total_silver FROM lakehouse.silver.payments_enriched").show()

spark.sql("""
SELECT
    payment_id,
    event_ts,
    card_id,
    status,
    tx_count_card_5m,
    distinct_merchants_card_10m,
    distinct_countries_card_1h,
    distinct_cards_device_1h,
    ROUND(declined_ratio_card_1h, 4) AS declined_ratio_card_1h
FROM lakehouse.silver.payments_enriched
ORDER BY event_ts DESC
LIMIT 10
""").show(truncate=False)

+------------+
|total_silver|
+------------+
|         179|
+------------+

+--------------------------------------------+--------------------------+----------+--------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|payment_id                                  |event_ts                  |card_id   |status  |tx_count_card_5m|distinct_merchants_card_10m|distinct_countries_card_1h|distinct_cards_device_1h|declined_ratio_card_1h|
+--------------------------------------------+--------------------------+----------+--------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|0fda00f5-e58f-47a3-afb2-dedc95643aa7_retry_1|2026-03-24 16:05:54.579228|CARD_00014|declined|6               |6                          |6                         |3                       |0.3333                |
|0fda00f5-e58f-47a3-afb2-dedc95643aa7_retry_0|2026-03-24 16:05:54.07